AP/CP fliter

In [ ]:
from tqdm import tqdm
import os
import pandas as pd
import csv
from multiprocessing import Pool, cpu_count

# ====================================================================
# 0. Helper: Safe ID normalization
# ====================================================================
def normalize_id(x):
    """Safely normalize MRN/PatientID strings."""
    x = str(x).strip()               # remove spaces
    x = x.replace(".0", "")          # remove float artifacts
    x = x.replace("\t", "").replace("\n", "")  # whitespace
    x = x.lstrip("0")                # remove leading zeros
    return x


# ====================================================================
# 2. Paths
# ====================================================================
csv_folder = r"<PRIVATE_DATA_PATH>"
csv_folder = r"<PRIVATE_DATA_PATH>"
clinical_file = "<PRIVATE_DATA_PATH>"
output_folder = "<PRIVATE_DATA_PATH>"
stats_output = "<PRIVATE_DATA_PATH>"

os.makedirs(output_folder, exist_ok=True)

csv_files = [f for f in os.listdir(csv_folder) if f.endswith('.csv')]


# ====================================================================
# 3. Load & clean clinical MRN table (global for workers)
# ====================================================================
clinical = pd.read_csv(
    clinical_file,
    usecols=['pat_id', 'PatientID'],
    dtype={'pat_id': str, 'PatientID': str},
    keep_default_na=False,
    low_memory=False,
)
clinical["PatientID"] = clinical["PatientID"].apply(normalize_id)
clinical["pat_id"] = clinical["pat_id"].astype(str)
clinical = clinical.drop_duplicates(subset=["PatientID"])

qualified_mrns = set(clinical["PatientID"])
print(f"Loaded {len(qualified_mrns)} qualified MRNs.")


# ====================================================================
# 4. Worker: process a single CSV and return stats rows
# ====================================================================
def process_single_csv(file_name: str):
    file_path = os.path.join(csv_folder, file_name)

    # read metadata safely
    file = pd.read_csv(
        file_path,
        dtype=str,
        keep_default_na=False,
        low_memory=False,
    )

    # STEP 1: Filter good slices
    if "slice_status" not in file.columns:
        print(f"❌ 'slice_status' missing in {file_name} → skipped.")
        return []   # no stats

    file = file[file["slice_status"] == "good"].copy()
    if file.empty:
        return []

    # STEP 0: Sort slices into correct order
    file["_dcm_sort_key"] = (
        file["dcm_name"].str.replace(".dcm", "", regex=False).astype(int)
    )

    file = (
        file.sort_values(["save_name", "_dcm_sort_key"])
            .assign(row_num=lambda df: df.groupby("save_name").cumcount())
            .drop(columns="_dcm_sort_key")
            .reset_index(drop=True)
    )

    # STEP 2: Normalize PatientID
    file["PatientID"] = file["PatientID"].apply(normalize_id)
    file = file[file["PatientID"].notna()]

    # STEP 3: Keep only patients in our clinical cohort
    file = file[file["PatientID"].isin(qualified_mrns)].copy()
    if file.empty:
        print(f"No matched MRN → skipped {file_name}")
        return []

    # STEP 4: Merge with clinical table to attach pat_id
    file = file.merge(clinical, on="PatientID", how="inner")
    if file["pat_id"].isna().any():
        raise ValueError(f"NaN pat_id after merge in {file_name}")

    # STEP 5: Per-(pat_id, tar_name, modality) slice counts
    if "Modality" not in file.columns:
        raise KeyError(f"Column 'Modality' not found in {file_name}.")

    per_csv_counts = file.groupby(["pat_id", "tar_name", "Modality"]).size()

    # Build stats rows for this CSV
    rows = []
    for (pat_id, tar_name, modality), n in per_csv_counts.items():
        rows.append(
            {
                "pat_id": str(pat_id),
                "tar_name": tar_name,
                "modality": modality,
                "num_slices": int(n),
            }
        )

    # STEP 6: Save per-slice metadata for this CSV
    output_path = os.path.join(output_folder, f"{file_name}_processed.csv")
    file.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)

    return rows


# ====================================================================
# 5. Run in parallel
# ====================================================================
if __name__ == "__main__":
    num_workers = min(10, cpu_count())  # tweak 8 → whatever you like

    all_rows = []

    with Pool(processes=num_workers) as pool:
        # tqdm over imap for progress bar
        for rows in tqdm(pool.imap_unordered(process_single_csv, csv_files),
                         total=len(csv_files),
                         desc="Processing CSVs in parallel"):
            if rows:
                all_rows.extend(rows)

    print("✅ All CSV files processed successfully! (good slices only, CSVs without 'slice_status' skipped)")

    # ====================================================================
    # 6. Dump global stats to CSV
    # ====================================================================
    if all_rows:
        stats_df = pd.DataFrame(all_rows)
        # if multiple entries per (pat_id, tar_name, modality), aggregate
        stats_df = (
            stats_df
            .groupby(["pat_id", "tar_name", "modality"], as_index=False)["num_slices"]
            .sum()
        )
        stats_df.to_csv(stats_output, index=False)
        print(f"📊 Saved slice-count table → {stats_output}")
    else:
        print("No stats collected (no matching patients).")


CT filter

In [ ]:
from tqdm import tqdm
import os
import pandas as pd
import csv
from multiprocessing import Pool, cpu_count

# ====================================================================
# Paths
# ====================================================================
csv_folder = r"<PRIVATE_DATA_PATH>"
clinical_file = "<PRIVATE_DATA_PATH>"
output_folder = "<PRIVATE_DATA_PATH>"
stats_output = "<PRIVATE_DATA_PATH>"

os.makedirs(output_folder, exist_ok=True)
csv_files = [f for f in os.listdir(csv_folder) if f.endswith(".csv")]

print(f"Found {len(csv_files)} metadata CSVs")


# ====================================================================
# Single-file worker
# ====================================================================
def process_one_csv(file_name):

    try:
        file_path = os.path.join(csv_folder, file_name)

        # Read metadata safely
        df = pd.read_csv(
            file_path,
            dtype=str,
            keep_default_na=False,
            low_memory=False,
        )

        # Keep CT
        if "Modality" not in df.columns:
            print(f"[ERROR] No Modality col: {file_name}")
            return []

        df = df[df["Modality"].isin(["CT"])].copy()
        if df.empty:
            print(f"[SKIP] No CT/MR in {file_name}")
            return []

        # Count slices
        counts = (
            df.groupby(["pat_id", "tar_name", "Modality"]).size()
        )

        rows = []
        for (pat_id, tar_name, modality), n in counts.items():
            rows.append({
                "pat_id": pat_id,
                "tar_name": tar_name,
                "modality": modality,
                "num_slices": int(n),
            })

        # Save filtered file
        out_path = os.path.join(output_folder, f"{file_name}_processed.csv")
        df.to_csv(out_path, index=False, quoting=csv.QUOTE_ALL)

        return rows

    except Exception as e:
        print(f"[ERROR] {file_name}: {e}")
        return []


# ====================================================================
# Main parallel execution
# ====================================================================
if __name__ == "__main__":

    num_workers = min(8, cpu_count())   # adjust based on GPFS load

    all_rows = []

    with Pool(processes=num_workers) as pool:
        for rows in tqdm(
            pool.imap_unordered(process_one_csv, csv_files, chunksize=1),
            total=len(csv_files),
            desc="Processing CSVs in parallel"
        ):
            if rows:
                all_rows.extend(rows)

    # ====================================================================
    # Build global slice-count table
    # ====================================================================
    if all_rows:
        stats_df = pd.DataFrame(all_rows)
        stats_df = (
            stats_df.groupby(["pat_id", "tar_name", "modality"], as_index=False)["num_slices"]
                    .sum()
        )
        stats_df.to_csv(stats_output, index=False)
        print(f"📊 Saved global CT/MR slice counts → {stats_output}")
    else:
        print("No stats collected.")


BodyPart/ImageType filter

In [ ]:
from tqdm import tqdm
import os
import pandas as pd
import csv
from multiprocessing import Pool, cpu_count

# ====================================================================
# Paths
# ====================================================================
csv_folder = r"<PRIVATE_DATA_PATH>"
output_folder = "<PRIVATE_DATA_PATH>"
stats_output = "<PRIVATE_DATA_PATH>"

os.makedirs(output_folder, exist_ok=True)
csv_files = [f for f in os.listdir(csv_folder) if f.endswith(".csv")]

print(f"Found {len(csv_files)} metadata CSVs")


# ====================================================================
# Single-file worker
# ====================================================================
def process_one_csv(file_name):

    try:
        file_path = os.path.join(csv_folder, file_name)

        # Read metadata safely
        df = pd.read_csv(
            file_path,
            dtype=str,
            keep_default_na=True,
            low_memory=False,
        )

        # Keep only relevant body regions
        keep_regions = {"PANCREAS", "ABDOMEN", "ABDOMEN+PELVIS", "CHABPE", "AORTA"}
        df = df[df["body_region"].isin(keep_regions)].copy()
        # Count slices
        counts = (
            df.groupby(["pat_id", "tar_name", "body_region"])
              .size()
              .reset_index(name="num_slices")
        )

        rows = counts.to_dict(orient="records")

        # Save filtered file
        out_path = os.path.join(output_folder, f"{file_name}_processed.csv")
        df.to_csv(out_path, index=False, quoting=csv.QUOTE_ALL)

        return rows

    except Exception as e:
        print(f"[ERROR] {file_name}: {e}")
        return []


# ====================================================================
# Main parallel execution
# ====================================================================
if __name__ == "__main__":

    num_workers = min(8, cpu_count())   # adjust based on GPFS load

    all_rows = []

    with Pool(processes=num_workers) as pool:
        for rows in tqdm(
            pool.imap_unordered(process_one_csv, csv_files, chunksize=1),
            total=len(csv_files),
            desc="Processing CSVs in parallel"
        ):
            if rows:
                all_rows.extend(rows)

    # ====================================================================
    # Build global slice-count table
    # ====================================================================
    if all_rows:
        stats_df = pd.DataFrame(all_rows)
        stats_df = (
            stats_df.groupby(
                ["pat_id", "tar_name", "body_region"],
                as_index=False
            )["num_slices"].sum()
        )
        stats_df.to_csv(stats_output, index=False)
        print(f"📊 Saved global CT slice counts → {stats_output}")
    else:
        print("No stats collected.")


In [ ]:
""